# ElevenLabs로 기본(clean) 음성 일괄 생성

`prompts.csv`의 한국어 프롬프트를 **억양(accent) × 화자(speaker)** 조합마다 ElevenLabs API로 읽혀서, 본 파이프라인 노트북이 바로 읽는 구조로 저장합니다.

```
audio_clean/{accent}/{speaker}/{prompt_id}.wav     ← 16 kHz, 모노, 16bit
```

**GPU 불필요**: 일반(CPU) 런타임에서 돌리세요.

## 준비물

1. **API 키**: Colab 왼쪽 🔑 *보안 비밀(Secrets)*에 `ELEVENLABS_API_KEY`라는 이름으로 등록하고 이 노트북의 접근 권한을 켜세요. 키를 코드나 노트북에 직접 적지 마세요.
2. **voices.csv** (Drive의 프로젝트 폴더): 억양·화자마다 사용할 ElevenLabs voice_id

```
accent,speaker,voice_id
ko,spk01,xxxxxxxxxxxxxxxxxxxx
zh,spk01,xxxxxxxxxxxxxxxxxxxx
zh,spk02,xxxxxxxxxxxxxxxxxxxx
```

   CSV 대신 아래 설정 셀의 `VOICES` 리스트에 직접 적어도 됩니다. voice_id는 ElevenLabs 웹의 보이스 설정이나 아래 4번 셀(보이스 목록)에서 확인할 수 있습니다.
3. **prompts.csv**: `prompt_id, prompt_text` (본 파이프라인과 같은 파일)

## 먼저 알아둘 것

- **콘텐츠 정책**: ElevenLabs는 불법 행위 지시, 혐오, 폭력 위협 등이 담긴 텍스트의 음성 생성을 정책으로 제한합니다. 유해 프롬프트를 대량으로 보내는 이 작업은 **일부 요청이 거부되거나 계정 조치 대상이 될 수 있습니다.** 시작 전에 ElevenLabs의 이용 정책을 직접 확인하고, 연구 목적 사용이 가능한지 문의해 보길 권합니다. 이 노트북은 거부된 프롬프트를 **우회하지 않고** 기록만 하며, 거부가 연속되면 자동으로 멈춥니다. 거부된 프롬프트는 분석에서 제외하세요(모든 조건에 공통으로 생성된 프롬프트만 사용). 우회하려고 문장을 바꾸면 실험 조건이 달라집니다.
- **크레딧**: 요금은 글자 수 기준입니다. 실행 전에 이 노트북이 총 글자 수를 계산해 보여 주니, 사용 중인 요금제의 잔여 크레딧과 비교한 뒤 `CONFIRM_RUN = True`로 바꾸세요.
- **재생성 금지 원칙**: 같은 문장도 호출할 때마다 음성이 조금씩 달라집니다. 마음에 안 든다고 다시 뽑아서 고르면 선택 편향이 생기니, 이미 만든 파일은 건너뛰도록 되어 있습니다. 다시 만들려면 파일을 직접 지우세요.
- **보이스 복제**: 실제 화자 음성(예: AIHub 녹음)으로 복제 보이스를 만든다면 데이터 이용 조건과 화자 동의, ElevenLabs의 복제 관련 약관을 확인하세요.

## 1. 설정

처음에는 `LIMIT_PROMPTS = 3`처럼 작게 시작해서 소리를 직접 들어 보세요.

In [ ]:
# ============================ 사용자 설정 ============================
DRY_RUN = False       # True: API 호출 없이 가짜 음성(톤)으로 전체 흐름만 점검

PROJECT_DIR = "/content/drive/MyDrive/ko_accent_jailbreak"
PROMPTS_CSV = f"{PROJECT_DIR}/prompts.csv"
VOICES_CSV  = f"{PROJECT_DIR}/voices.csv"          # 있으면 이 파일을 우선 사용
AUDIO_DIR   = f"{PROJECT_DIR}/audio_clean"         # 본 파이프라인의 AUDIO_DIR과 같게

# voices.csv가 없을 때 사용 (accent, speaker, voice_id)
VOICES = [
    {"accent": "ko", "speaker": "spk01", "voice_id": "PASTE_VOICE_ID"},
    {"accent": "zh", "speaker": "spk01", "voice_id": "PASTE_VOICE_ID"},
]

MODEL_ID      = "eleven_multilingual_v2"   # 모델별 지원 언어·크레딧 단가는 ElevenLabs 문서에서 확인
OUTPUT_FORMAT = "pcm_16000"                # pcm_16000 권장 (44.1kHz PCM은 상위 요금제 필요)
VOICE_SETTINGS = dict(stability=0.5, similarity_boost=0.75, style=0.0, use_speaker_boost=True)
USE_SEED = True      # 같은 입력에 최대한 같은 결과가 나오도록 시도 (보장되지는 않음)
SEED = 42
LANGUAGE_CODE = None # 예: "ko". 일부 모델은 이 값을 지원하지 않으니 기본은 None(자동 감지)

LIMIT_PROMPTS  = 3      # 처음 N개 프롬프트만. None이면 전체
ONLY_ACCENTS   = None   # 예: ["zh", "ja"]. None이면 voices의 모든 억양
VALIDATE_VOICES = True  # 실행 전에 voice_id가 실제로 있는지, 라벨(억양 등)은 무엇인지 확인
LIST_MY_VOICES  = False # True면 내 계정의 보이스 목록을 출력 (voice_id 찾을 때)

CONFIRM_RUN = False     # 계획과 글자 수를 확인한 뒤 True로 바꿔서 실제 생성
RETRY_REJECTED = False  # 이전에 거부된 프롬프트를 다시 시도할지 (원인을 해결한 뒤에만)
SLEEP_SEC = 0.3         # 호출 사이 대기 (요금제의 동시 요청·속도 제한 보호)
MAX_RETRIES = 5         # 429(속도 제한), 5xx 오류 재시도 횟수
MAX_CONSECUTIVE_REJECTS = 5   # 서로 다른 프롬프트가 연속으로 거부되면 중단
# ===================================================================

In [ ]:
import os, sys, io, re, json, time, random, datetime
IN_COLAB = "google.colab" in sys.modules

if DRY_RUN:
    PROJECT_DIR = "/content/kaj_tts_dryrun" if IN_COLAB else os.path.abspath("./kaj_tts_dryrun")
    PROMPTS_CSV = f"{PROJECT_DIR}/prompts.csv"
    VOICES_CSV  = f"{PROJECT_DIR}/voices.csv"
    AUDIO_DIR   = f"{PROJECT_DIR}/audio_clean"
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

LOG_DIR = f"{PROJECT_DIR}/tts_logs"
for d in (PROJECT_DIR, AUDIO_DIR, LOG_DIR):
    os.makedirs(d, exist_ok=True)
LOG_PATH = f"{LOG_DIR}/tts_log.jsonl"

import numpy as np, pandas as pd, soundfile as sf, librosa, requests
from tqdm.auto import tqdm

SR_OUT = 16000
SAFE_NAME = re.compile(r"^[A-Za-z0-9_.\-]+$")

def read_csv_ko(path, **kw):
    # 한국어 윈도우 엑셀 'CSV'는 CP949로 저장됩니다. 순서대로 시도합니다.
    for enc in ("utf-8-sig", "cp949", "euc-kr"):
        try:
            return pd.read_csv(path, encoding=enc, **kw)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"{path}의 인코딩을 알 수 없습니다. 'CSV UTF-8'로 다시 저장해 주세요.")

def read_jsonl(path):
    rows = []
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return rows

def append_jsonl(path, row):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()

def get_api_key():
    # 키는 출력하거나 로그에 남기지 않습니다.
    try:
        from google.colab import userdata
        k = userdata.get("ELEVENLABS_API_KEY")
        if k:
            return k
    except Exception:
        pass
    k = os.environ.get("ELEVENLABS_API_KEY")
    if k:
        return k
    import getpass
    return getpass.getpass("ElevenLabs API 키 (화면에 표시되지 않음): ")

API_KEY = "dry-run" if DRY_RUN else get_api_key()
assert API_KEY, "API 키가 없습니다. Colab Secrets에 ELEVENLABS_API_KEY를 등록하세요."
print("준비 완료 | DRY_RUN:", DRY_RUN, "| 로그:", LOG_PATH)

## 2. (선택) 내 계정 보이스 목록

`voice_id`를 찾을 때 `LIST_MY_VOICES = True`로 두고 이 셀을 실행하세요. 이름, ID, 라벨(억양·언어 등)을 보여 줍니다. 라벨은 보이스 제작자가 붙인 것이라 **실제 억양과 다를 수 있으니 반드시 직접 들어 보세요.**

In [ ]:
API_BASE = "https://api.elevenlabs.io"

def api_headers():
    return {"xi-api-key": API_KEY, "Content-Type": "application/json"}

if LIST_MY_VOICES and not DRY_RUN:
    try:
        r = requests.get(f"{API_BASE}/v1/voices", headers=api_headers(), timeout=60)
        r.raise_for_status()
        for v in r.json().get("voices", []):
            print(f'{v.get("name")!s:30} {v.get("voice_id")}  {v.get("labels")}')
    except Exception as e:
        print("보이스 목록을 불러오지 못했습니다:", repr(e)[:200])
        print("→ ElevenLabs 웹에서 보이스의 ID를 복사해 voices.csv에 넣으세요.")
else:
    print("건너뜀 (LIST_MY_VOICES=False)")

## 3. 입력 불러오기 · 생성 계획

`DRY_RUN`일 때는 무해한 자리표시 문장과 가짜 voice_id로 입력을 만듭니다.

In [ ]:
if DRY_RUN:
    pd.DataFrame({"prompt_id": [f"p{i:04d}" for i in range(6)],
                  "prompt_text": [f"테스트 문장 {i}번입니다. 오늘 저녁 메뉴를 추천해 주세요." + "네" * i for i in range(6)]}
                 ).to_csv(PROMPTS_CSV, index=False, encoding="cp949")   # CP949 읽기도 함께 점검
    pd.DataFrame([{"accent": a, "speaker": s, "voice_id": f"dry_{a}_{s}"}
                  for a in ("ko", "zh", "ja") for s in ("spk01", "spk02")]
                 ).to_csv(VOICES_CSV, index=False)

prompts = read_csv_ko(PROMPTS_CSV, dtype=str)
prompts.columns = prompts.columns.str.strip()
assert {"prompt_id", "prompt_text"} <= set(prompts.columns), "prompts.csv에 prompt_id, prompt_text 컬럼이 필요합니다."
prompts["prompt_id"] = prompts["prompt_id"].str.strip()
prompts["prompt_text"] = prompts["prompt_text"].fillna("").str.strip()
dup = prompts.prompt_id.duplicated().sum()
if dup:
    print(f"⚠️ 중복 prompt_id {dup}개는 첫 번째만 사용합니다.")
    prompts = prompts.drop_duplicates("prompt_id")
empty = (prompts.prompt_text == "")
if empty.any():
    print(f"⚠️ 빈 텍스트 {empty.sum()}개는 제외합니다.")
    prompts = prompts[~empty]
if LIMIT_PROMPTS:
    prompts = prompts.head(LIMIT_PROMPTS)
prompts = prompts.reset_index(drop=True)

voices = read_csv_ko(VOICES_CSV, dtype=str) if os.path.exists(VOICES_CSV) else pd.DataFrame(VOICES)
voices.columns = voices.columns.str.strip()
assert {"accent", "speaker", "voice_id"} <= set(voices.columns), "voices에 accent, speaker, voice_id가 필요합니다."
for c in ("accent", "speaker", "voice_id"):
    voices[c] = voices[c].astype(str).str.strip()
if ONLY_ACCENTS:
    voices = voices[voices.accent.isin(ONLY_ACCENTS)]
if not DRY_RUN:
    bad = voices[voices.voice_id.str.startswith("PASTE") | (voices.voice_id == "")]
    assert bad.empty, "voice_id가 비어 있거나 자리표시자(PASTE_VOICE_ID)입니다: " + ", ".join(bad.accent + "/" + bad.speaker)
for name in list(voices.accent) + list(voices.speaker) + list(prompts.prompt_id):
    assert SAFE_NAME.match(name), f"폴더·파일명에 쓸 수 없는 문자가 있습니다: {name!r} (영문, 숫자, _ . - 만 허용)"
assert not voices.duplicated(["accent", "speaker"]).any(), "같은 (accent, speaker)가 voices에 중복되어 있습니다."

# 생성 계획: 프롬프트 × (억양, 화자)
plan = pd.DataFrame([
    {"prompt_id": p.prompt_id, "text": p.prompt_text, "accent": v.accent, "speaker": v.speaker, "voice_id": v.voice_id,
     "path": f"{AUDIO_DIR}/{v.accent}/{v.speaker}/{p.prompt_id}.wav"}
    for p in prompts.itertuples() for v in voices.itertuples()
])
plan["exists"] = plan.path.apply(os.path.exists)
plan["chars"] = plan.text.str.len()

# 이전에 거부된 프롬프트는 기본적으로 다시 보내지 않음 (같은 텍스트는 같은 결과일 가능성이 높고, 반복 전송은 정책상 불리).
# voice_id를 고치는 등 원인을 해결했다면 RETRY_REJECTED = True로 다시 시도하세요.
prev = read_jsonl(LOG_PATH)
rejected_before = {r["prompt_id"] for r in prev if r.get("status") == "rejected"}   # 문장 기준 (보이스 무관)
plan["prev_rejected"] = plan.prompt_id.isin(rejected_before)

pending = plan[~plan.exists & (RETRY_REJECTED | ~plan.prev_rejected)]
print(f"프롬프트 {len(prompts)}개 × 조건 {len(voices)}개 = 총 {len(plan)}개")
print(f"  이미 있음 {int(plan.exists.sum())}개 | 이전 거부로 건너뜀 {int((~plan.exists & plan.prev_rejected).sum()) if not RETRY_REJECTED else 0}개")
print(f"  이번에 생성할 파일 {len(pending)}개 | 사용 글자 수 약 {int(pending.chars.sum()):,}자")
print("\n조건 목록:")
print(voices.groupby("accent").speaker.apply(lambda s: ", ".join(s)).to_string())

In [ ]:
# voice_id가 실제로 존재하는지, 라벨이 무엇인지 확인 (실제 생성 전 점검)
if VALIDATE_VOICES and not DRY_RUN:
    for v in voices.itertuples():
        try:
            r = requests.get(f"{API_BASE}/v1/voices/{v.voice_id}", headers=api_headers(), timeout=30)
            if r.status_code == 200:
                j = r.json()
                print(f"✅ {v.accent}/{v.speaker}: {j.get('name')} | 라벨 {j.get('labels')}")
            else:
                print(f"⚠️ {v.accent}/{v.speaker}: voice_id 확인 실패 (HTTP {r.status_code})")
        except Exception as e:
            print(f"⚠️ {v.accent}/{v.speaker}: 확인 중 오류 {repr(e)[:120]}")
else:
    print("건너뜀")

## 4. 생성

- 이미 만든 파일은 건너뛰고(이어하기), 요청마다 결과를 `tts_log.jsonl`에 기록합니다. 로그에는 프롬프트 원문이나 API 키를 남기지 않고, ID와 상태, 글자 수만 남깁니다.
- 속도 제한(429)과 서버 오류(5xx)는 점점 길게 기다리며 재시도합니다.
- 키가 틀렸거나 크레딧이 소진되면 즉시 멈춥니다.
- 정책 등으로 거부된 프롬프트는 같은 프롬프트의 나머지 조건에서도 보내지 않고, 서로 다른 프롬프트가 `MAX_CONSECUTIVE_REJECTS`번 연속 거부되면 전체를 멈춥니다.
- PCM/WAV 응답은 16 kHz 모노 16bit WAV로 정리합니다(다른 샘플링 레이트는 리샘플).

In [ ]:
class AbortRun(Exception):
    pass

def audio_bytes_to_wav(content, out_path):
    # RIFF 헤더가 있으면 WAV로, 없으면 raw 16bit PCM으로 처리
    if content[:4] == b"RIFF":
        y, sr = sf.read(io.BytesIO(content), dtype="float32")
    else:
        m = re.search(r"pcm_(\d+)", OUTPUT_FORMAT)
        if not m:
            raise ValueError(f"OUTPUT_FORMAT={OUTPUT_FORMAT}: pcm_* 또는 WAV 응답만 지원합니다.")
        sr = int(m.group(1))
        if len(content) % 2:
            content = content[:-1]
        y = np.frombuffer(content, dtype="<i2").astype(np.float32) / 32768.0
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != SR_OUT:
        y = librosa.resample(y, orig_sr=sr, target_sr=SR_OUT)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    tmp = out_path + ".part"
    sf.write(tmp, y, SR_OUT, subtype="PCM_16", format="WAV")
    os.replace(tmp, out_path)       # 중간에 끊겨도 반쪽 파일이 남지 않게
    return len(y) / SR_OUT

def fake_audio(text):
    # DRY_RUN: 글자 수에 비례한 길이의 톤을 raw PCM으로 생성
    n = int(SR_OUT * max(0.5, len(text) * 0.12))
    t = np.arange(n) / SR_OUT
    return (0.3 * np.sin(2 * np.pi * 220 * t) * 32767).astype("<i2").tobytes()

def tts_request(voice_id, text):
    # 반환: dict(kind, status, content, request_id, message)
    if DRY_RUN:
        return {"kind": "ok", "status": 200, "content": fake_audio(text), "request_id": "dry", "message": ""}
    body = {"text": text, "model_id": MODEL_ID, "voice_settings": VOICE_SETTINGS}
    if USE_SEED:
        body["seed"] = SEED
    if LANGUAGE_CODE:
        body["language_code"] = LANGUAGE_CODE
    url = f"{API_BASE}/v1/text-to-speech/{voice_id}"
    last = {"kind": "failed", "status": None, "content": b"", "request_id": None, "message": ""}
    for attempt in range(MAX_RETRIES + 1):
        try:
            r = requests.post(url, headers=api_headers(), params={"output_format": OUTPUT_FORMAT},
                              json=body, timeout=120)
        except requests.RequestException as e:
            last["message"] = f"network: {type(e).__name__}"
            time.sleep(min(60, 2 ** attempt * 2))
            continue
        rid = r.headers.get("request-id")
        snippet = r.text[:200].replace("\n", " ")
        if r.status_code == 200:
            return {"kind": "ok", "status": 200, "content": r.content, "request_id": rid, "message": ""}
        last = {"kind": "failed", "status": r.status_code, "content": b"", "request_id": rid, "message": snippet}
        low = snippet.lower()
        if r.status_code in (401, 402) or "quota" in low:
            return {**last, "kind": "quota" if "quota" in low or r.status_code == 402 else "auth"}
        if r.status_code == 429 or r.status_code >= 500:
            wait = r.headers.get("retry-after")
            wait = float(wait) if wait and wait.replace(".", "").isdigit() else min(60, 2 ** attempt * 2)
            time.sleep(wait)
            continue
        return {**last, "kind": "rejected"}     # 그 밖의 4xx: 재시도해도 같은 결과
    return last

def run_generation():
    tracker = {"ok": 0, "rejected": 0, "failed": 0, "skipped": 0}
    consecutive_rejects = 0
    todo = pending.copy()
    for pid, grp in tqdm(todo.groupby("prompt_id", sort=False), total=todo.prompt_id.nunique(), desc="프롬프트"):
        prompt_rejected = False
        any_reject_this_prompt = False
        for row in grp.itertuples():
            if prompt_rejected:
                tracker["skipped"] += 1
                append_jsonl(LOG_PATH, {"ts": datetime.datetime.now().isoformat(timespec="seconds"),
                                        "prompt_id": row.prompt_id, "accent": row.accent, "speaker": row.speaker,
                                        "voice_id": row.voice_id, "status": "skipped_after_reject", "chars": row.chars})
                continue
            t0 = time.time()
            res = tts_request(row.voice_id, row.text)
            entry = {"ts": datetime.datetime.now().isoformat(timespec="seconds"), "prompt_id": row.prompt_id,
                     "accent": row.accent, "speaker": row.speaker, "voice_id": row.voice_id,
                     "model_id": MODEL_ID, "output_format": OUTPUT_FORMAT, "chars": row.chars,
                     "http_status": res["status"], "request_id": res["request_id"]}
            if res["kind"] == "ok":
                try:
                    dur = audio_bytes_to_wav(res["content"], row.path)
                    entry.update(status="ok", duration_s=round(dur, 2), sec=round(time.time() - t0, 2))
                    tracker["ok"] += 1
                    consecutive_rejects = 0
                except Exception as e:
                    entry.update(status="failed", error=f"decode: {repr(e)[:150]}")
                    tracker["failed"] += 1
            elif res["kind"] in ("auth", "quota"):
                entry.update(status=res["kind"], error=res["message"])
                append_jsonl(LOG_PATH, entry)
                raise AbortRun("API 키가 올바르지 않거나 크레딧이 부족합니다." if res["kind"] == "auth"
                               else "크레딧(쿼터)이 부족합니다. 요금제를 확인하세요.")
            elif res["kind"] == "rejected":
                entry.update(status="rejected", error=res["message"])
                tracker["rejected"] += 1
                prompt_rejected = True
                any_reject_this_prompt = True
            else:
                entry.update(status="failed", error=res["message"])
                tracker["failed"] += 1
            append_jsonl(LOG_PATH, entry)
            time.sleep(SLEEP_SEC if not DRY_RUN else 0)
        if any_reject_this_prompt:
            consecutive_rejects += 1
            if consecutive_rejects >= MAX_CONSECUTIVE_REJECTS:
                print(f"\n⛔ 서로 다른 프롬프트 {consecutive_rejects}개가 연속으로 거부되어 중단합니다.")
                print("   마지막 오류:", entry.get("error", "")[:200])
                print("   → 이용 정책, voice_id, 모델 설정을 확인하세요. 거부된 프롬프트를 고쳐서 다시 보내지는 마세요(실험 조건이 달라집니다).")
                break
    return tracker

if len(pending) == 0:
    print("생성할 파일이 없습니다.")
elif not (CONFIRM_RUN or DRY_RUN):
    print(f"⏸ 아직 실행하지 않았습니다. 위의 계획(파일 {len(pending)}개, 약 {int(pending.chars.sum()):,}자)과 잔여 크레딧을 확인한 뒤")
    print("   설정 셀에서 CONFIRM_RUN = True로 바꾸고 이 셀부터 다시 실행하세요.")
else:
    try:
        result = run_generation()
        print("\n결과:", result)
    except AbortRun as e:
        print("\n⛔ 중단:", e)

## 5. 점검: 개수 · 길이 · 말속도 · 공통 프롬프트

- **조건별 말속도(글자/초)**: 억양마다 말하는 속도가 크게 다르면, 억양 효과와 속도 효과가 섞입니다. 억양 연구에서는 같이 보고하는 것이 좋습니다.
- **공통 프롬프트**: 본 파이프라인은 **모든 (억양, 화자) 조건에 공통으로 있는 프롬프트**만 씁니다. 일부 조건에서 실패한 프롬프트가 있으면 여기서 줄어듭니다.

In [ ]:
plan["exists"] = plan.path.apply(os.path.exists)
done = plan[plan.exists].copy()
print(f"생성된 파일 {len(done)} / 계획 {len(plan)}")

if len(done):
    info = []
    for r in done.itertuples():
        try:
            y, sr = sf.read(r.path, dtype="float32")
            info.append((len(y) / sr, float(np.max(np.abs(y))) if len(y) else 0.0,
                         float(np.sqrt(np.mean(y ** 2))) if len(y) else 0.0))
        except Exception:
            info.append((np.nan, np.nan, np.nan))
    done[["dur", "peak", "rms"]] = info
    done["cps"] = done.chars / done.dur            # 글자/초

    flag = done[(done.dur < 0.5) | (done.peak < 0.01) | done.dur.isna()]
    if len(flag):
        print(f"\n⚠️ 너무 짧거나 거의 무음인 파일 {len(flag)}개 (직접 들어 보세요):")
        print(flag[["accent", "speaker", "prompt_id", "dur", "peak"]].to_string(index=False))

    print("\n조건별 평균 (길이 초, 말속도 글자/초):")
    print(done.groupby(["accent", "speaker"]).agg(n=("path", "size"), dur=("dur", "mean"), cps=("cps", "mean")).round(2).to_string())

    conds = plan[["accent", "speaker"]].drop_duplicates()
    have = done.groupby(["accent", "speaker"]).prompt_id.apply(set)
    full = set.intersection(*[have.get((a, s), set()) for a, s in conds.itertuples(index=False)])
    print(f"\n모든 조건({len(conds)}개)에 공통으로 있는 프롬프트: {len(full)} / {plan.prompt_id.nunique()}개")
    missing = sorted(set(plan.prompt_id) - full)
    if missing:
        print("공통에서 빠진 prompt_id:", ", ".join(missing[:20]) + (" ..." if len(missing) > 20 else ""))

# 로그 요약 (이번 입력 범위의 최신 상태)
log = pd.DataFrame(read_jsonl(LOG_PATH))
if len(log):
    print("\n로그 상태별 건수:")
    print(log.status.value_counts().to_string())

In [ ]:
# 실행 설정과 결과를 재현용으로 저장 (API 키는 포함하지 않음)
meta = {"model_id": MODEL_ID, "output_format": OUTPUT_FORMAT, "voice_settings": VOICE_SETTINGS,
        "seed": SEED if USE_SEED else None, "language_code": LANGUAGE_CODE,
        "saved_at": datetime.datetime.now().isoformat(timespec="seconds"), "dry_run": DRY_RUN}
with open(f"{LOG_DIR}/tts_settings.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

if len(log):
    ok = log[log.status == "ok"].drop_duplicates(["accent", "speaker", "prompt_id"], keep="last")
    cols = [c for c in ["accent", "speaker", "prompt_id", "voice_id", "model_id", "output_format", "chars",
                        "duration_s", "request_id", "ts"] if c in ok.columns]
    ok[cols].to_csv(f"{LOG_DIR}/tts_manifest.csv", index=False, encoding="utf-8-sig")
    print(f"저장: {LOG_DIR}/tts_manifest.csv ({len(ok)}행), tts_settings.json")

In [ ]:
# 조건별 첫 파일 들어 보기 (억양이 의도대로 들리는지 직접 확인)
from IPython.display import Audio, display
for (acc, spk), g in done.groupby(["accent", "speaker"]):
    r = g.sort_values("prompt_id").iloc[0]
    print(f"{acc}/{spk}/{r.prompt_id}  ({r.dur:.1f}초)")
    display(Audio(r.path))

## 다음 단계

1. 위에서 직접 들어 보고, 억양이 의도대로 들리는지 소수 인원이 평가해 두면 논문에 쓸 근거가 됩니다.
2. 본 파이프라인 노트북의 `AUDIO_DIR`을 이 노트북의 `AUDIO_DIR`과 같게 두고(`.../audio_clean`) 실행하세요. `metadata.csv`는 만들지 않으므로 폴더 구조로 자동 인식됩니다.
3. 프롬프트·화자를 늘릴 때는 `LIMIT_PROMPTS`를 키우고 `CONFIRM_RUN`을 확인한 뒤 다시 실행하세요. 이미 만든 파일은 건너뜁니다.